In [1]:
#https://pygithub.readthedocs.io/en/stable/introduction.html
from github import Github
# Authentication is defined via github.Auth
from github import Auth
import pandas as pd
import numpy as np
import json 
from datetime import datetime, date
from collections import Counter
import plotly.express as px
import time
import pickle
intrinsic_people = ["@aaronchongth","@akash-roboticist","@andreasBihlmaier","@arjo129","@audrow","@azeey","@damon-oss","@faximan","@jennuine","@koonpeng","@kscottz","@luca-della-vedova","@marcoag","@mbordignon-intrinsic","@methylDragon","@mjcarroll","@mjeronimo","@mxgrey","@nuclearsandwich-ai","@quarkytale","@scpeters","@sloretz","@tfoote","@udaya2899","@xiyuoh","@Yadunund"]
org_name = "open-rmf"
this_year = 2024
last_year = 2023

In [2]:
# Grab the access token
with open('./tokens.json',"r") as json_data:
    tokens = json.loads(json_data.read())
    json_data.close()

auth = Auth.Token(tokens["Github"])

# Public Web Github
gh = Github(auth=auth)

In [3]:
org = gh.get_organization(org_name)
repos = org.get_repos()

In [4]:
def extract_contributions(gh, repo_name, start_date, end_date):
# Get github repo level stats for two date ranges
    repo = gh.get_repo(repo_name)
    prs = repo.get_pulls(state='closed', sort='created')
    year = []
    results = {}
    count = 0
    for pr in prs:
        if start_date < pr.closed_at.date() < end_date:
            year.append(pr)
            count += 1
            
    results["repo"] = repo_name
    results["prs"] = year
    results["start_date"] = start_date    
    results["end_date"] = end_date    
    results["users"] = []
    results["handles"] = []
    results["add"] = 0
    results["del"] = 0 
    results["comments"] = 0
    results["files"] = 0
    
    for pr in year:
        results["users"].append(pr.user.name)
        results["handles"].append(pr.user.login)
        results["files"] += pr.changed_files 
        results["del"] += pr.deletions
        results["add"] += pr.additions
        results["comments"] += pr.review_comments

    results["total_prs"] = count
    results["total_users"] = len(set(results["users"]))

    return results

In [5]:
# Create a list of github repos for an org
full_repo_list = []
i = 0
has_repos = True
while has_repos:
    next_repos = repos.get_page(i)
    if len(next_repos) > 0:
        i += 1
        full_repo_list += next_repos
    else:
        has_repos = False
        
print(full_repo_list)
print(len(full_repo_list))

[Repository(full_name="open-rmf/rmf_traffic_editor"), Repository(full_name="open-rmf/free_fleet"), Repository(full_name="open-rmf/rmf-web"), Repository(full_name="open-rmf/traffic_editor_assets"), Repository(full_name="open-rmf/menge_vendor"), Repository(full_name="open-rmf/fleet_adapter_mir"), Repository(full_name="open-rmf/free_fleet_ros2"), Repository(full_name="open-rmf/free_fleet_ros1"), Repository(full_name="open-rmf/rmf-cloud-tools"), Repository(full_name="open-rmf/rmf_simulation"), Repository(full_name="open-rmf/rmf"), Repository(full_name="open-rmf/.github"), Repository(full_name="open-rmf/rmf_demos"), Repository(full_name="open-rmf/rmf_internal_msgs"), Repository(full_name="open-rmf/rmf_battery"), Repository(full_name="open-rmf/rmf_task"), Repository(full_name="open-rmf/rmf_traffic"), Repository(full_name="open-rmf/rmf_ros2"), Repository(full_name="open-rmf/rmf_utils"), Repository(full_name="open-rmf/rmf_cmake_uncrustify"), Repository(full_name="open-rmf/ament_cmake_catch2"),

In [6]:
this_year_start = date(2025, 3, 11)
this_year_end = date(2025, 9, 11)
last_year_start = date(2024, 9, 10)
last_year_end = date(2025, 3, 10)
full_org_results = {}
fname = 'github_stats_{0}_{1}-{2}.pkl'.format(org_name,this_year,last_year)
for repo in full_repo_list:
    print("Extracting data for {0} from {1} to {2}".format(repo.full_name,this_year_start,this_year_end))
    this_year_results = extract_contributions(gh, repo.full_name, this_year_start, this_year_end)
    print("Extracting data for {0} from {1} to {2}".format(repo.full_name,last_year_start,last_year_end))
    last_year_results = extract_contributions(gh, repo.full_name, last_year_start, last_year_end)
    full_org_results[repo.name] = {}
    full_org_results[repo.name][this_year] = this_year_results
    full_org_results[repo.name][last_year] = last_year_results
    with open(fname,"wb") as file:
        pickle.dump(full_org_results, file)
        print("Wrote: {0}".format(fname))
    print("-----------------------------")



Extracting data for open-rmf/rmf_traffic_editor from 2025-03-11 to 2025-09-11
Extracting data for open-rmf/rmf_traffic_editor from 2024-09-10 to 2025-03-10
Wrote: github_stats_open-rmf_2024-2023.pkl
-----------------------------
Extracting data for open-rmf/free_fleet from 2025-03-11 to 2025-09-11
Extracting data for open-rmf/free_fleet from 2024-09-10 to 2025-03-10
Wrote: github_stats_open-rmf_2024-2023.pkl
-----------------------------
Extracting data for open-rmf/rmf-web from 2025-03-11 to 2025-09-11
Extracting data for open-rmf/rmf-web from 2024-09-10 to 2025-03-10
Wrote: github_stats_open-rmf_2024-2023.pkl
-----------------------------
Extracting data for open-rmf/traffic_editor_assets from 2025-03-11 to 2025-09-11
Extracting data for open-rmf/traffic_editor_assets from 2024-09-10 to 2025-03-10
Wrote: github_stats_open-rmf_2024-2023.pkl
-----------------------------
Extracting data for open-rmf/menge_vendor from 2025-03-11 to 2025-09-11
Extracting data for open-rmf/menge_vendor fr

Extracting data for open-rmf/webrtc-robot-gateway from 2024-09-10 to 2025-03-10
Wrote: github_stats_open-rmf_2024-2023.pkl
-----------------------------
Extracting data for open-rmf/lift_adapter_template from 2025-03-11 to 2025-09-11
Extracting data for open-rmf/lift_adapter_template from 2024-09-10 to 2025-03-10
Wrote: github_stats_open-rmf_2024-2023.pkl
-----------------------------
Extracting data for open-rmf/rmf-arkit-ios from 2025-03-11 to 2025-09-11
Extracting data for open-rmf/rmf-arkit-ios from 2024-09-10 to 2025-03-10
Wrote: github_stats_open-rmf_2024-2023.pkl
-----------------------------
Extracting data for open-rmf/rmf-gateway-pi from 2025-03-11 to 2025-09-11
Extracting data for open-rmf/rmf-gateway-pi from 2024-09-10 to 2025-03-10
Wrote: github_stats_open-rmf_2024-2023.pkl
-----------------------------
Extracting data for open-rmf/rmf-panel-js from 2025-03-11 to 2025-09-11
Extracting data for open-rmf/rmf-panel-js from 2024-09-10 to 2025-03-10
Wrote: github_stats_open-rmf

Wrote: github_stats_open-rmf_2024-2023.pkl
-----------------------------
Extracting data for open-rmf/rmf_workcell from 2025-03-11 to 2025-09-11
Extracting data for open-rmf/rmf_workcell from 2024-09-10 to 2025-03-10
Wrote: github_stats_open-rmf_2024-2023.pkl
-----------------------------
Extracting data for open-rmf/ros2-impulse-examples from 2025-03-11 to 2025-09-11
Extracting data for open-rmf/ros2-impulse-examples from 2024-09-10 to 2025-03-10
Wrote: github_stats_open-rmf_2024-2023.pkl
-----------------------------
Extracting data for open-rmf/next_gen_prototype from 2025-03-11 to 2025-09-11
Extracting data for open-rmf/next_gen_prototype from 2024-09-10 to 2025-03-10
Wrote: github_stats_open-rmf_2024-2023.pkl
-----------------------------


In [7]:
out =  None
with open(fname, 'rb') as file:
        out = pickle.load(file)      
print(len(out.keys()))
print(out.keys())

77
dict_keys(['rmf_traffic_editor', 'free_fleet', 'rmf-web', 'traffic_editor_assets', 'menge_vendor', 'fleet_adapter_mir', 'free_fleet_ros2', 'free_fleet_ros1', 'rmf-cloud-tools', 'rmf_simulation', 'rmf', '.github', 'rmf_demos', 'rmf_internal_msgs', 'rmf_battery', 'rmf_task', 'rmf_traffic', 'rmf_ros2', 'rmf_utils', 'rmf_cmake_uncrustify', 'ament_cmake_catch2', 'traffic-editor-js', 'rmf_docs', 'rmf_visualization_msgs', 'rmf_visualization', 'ts_ros', 'rmf_building_map_msgs', 'mock-lift-io', 'stubborn_buddies', 'ros2-bridge', 'rmf_gym', 'magni_nav_ros1', 'fleet_adapter_template', 'door_adapter_template', 'rmf_fullstack_installer', 'rmf-gateway-app', 'rmf_massrobotics', 'webrtc-robot-gateway', 'lift_adapter_template', 'rmf-arkit-ios', 'rmf-gateway-pi', 'rmf-panel-js', 'rmf_geometry_testbed', 'rmf_api_msgs', 'nlohmann_json_schema_validator_vendor', 'traffic_editor_iii', 'rmf_integration_tests', 'TemiFleetAdapterBridge', 'temi_fleet_adapter_python', 'rmf_rtls', 'pybind11_json_vendor', 'temi-

In [8]:
# Do full org aggregation
to_agg = ["users","add","del","files"]

full_results = {}
full_results[this_year] = {}
full_results[last_year] = {}

first = True
for key in full_org_results.keys():
    if first:
        full_results[this_year] = {k: full_org_results[key][this_year][k] for k in to_agg}
        full_results[last_year] = {k: full_org_results[key][last_year][k] for k in to_agg}
        full_results[this_year]["prs"] = len(full_org_results[key][this_year]["prs"])
        full_results[last_year]["prs"] = len(full_org_results[key][last_year]["prs"])
        first = False
    else:        
        for a in to_agg:
            full_results[this_year][a] += full_org_results[key][this_year][a]
            full_results[last_year][a] += full_org_results[key][last_year][a]
            full_results[this_year]["prs"] += len(full_org_results[key][this_year]["prs"])
            full_results[last_year]["prs"] += len(full_org_results[key][last_year]["prs"])

full_results[last_year]["contributors"] = set(full_results[last_year]["users"])
full_results[last_year]["users"] = len(set(full_results[last_year]["users"]))

full_results[this_year]["contributors"] = set(full_results[this_year]["users"])
full_results[this_year]["users"] = len(set(full_results[this_year]["users"]))


print("Results for {0} ==> {1}".format(last_year,this_year))
print("-------------------------")

temp_this = {}
temp_last = {}
temp_change = {}

for k in full_results[this_year].keys():
    if k == "contributors":
        continue
    change = -100*(full_results[last_year][k]-full_results[this_year][k])/full_results[last_year][k]
    temp_this[k] =  full_results[this_year][k]
    temp_last[k] =  full_results[last_year][k]
    temp_change[k] = change
    print("{0:6s}| {1} : {2:<6} | {3} : {4:<6} | {5:4.2f}%".format(k,last_year,full_results[last_year][k],this_year,full_results[this_year][k],change))
print(full_results[this_year]["contributors"])

summary_results = pd.DataFrame(data=[temp_this,temp_last,temp_change])
summary_results.to_csv("{0}-{1}-{2}-GithubContribsSummary.csv".format(org_name,last_year,this_year))

Results for 2023 ==> 2024
-------------------------
users | 2023 : 21     | 2024 : 22     | 4.76%
add   | 2023 : 171969 | 2024 : 2855422 | 1560.43%
del   | 2023 : 81539  | 2024 : 45519  | -44.18%
files | 2023 : 2313   | 2024 : 5592   | 141.76%
prs   | 2023 : 647    | 2024 : 1169   | 80.68%
{'Florian', 'Jun', None, 'Aaron Chong', 'Alejandro Hernández Cordero', 'Shamly Mackey', 'Francesco Fallica', 'yadunund', 'Teo Koon Peng', 'Gary Bey', 'Luca Della Vedova', 'Saurabh Kamat', 'The Construct', 'Cheng-Wei Chen', 'Pedro de Azeredo', 'kj', 'Xiyu', 'Akshaya Katiganere Anandappa', 'Arjo Chakravarty', 'John TGZ', 'Grey', 'kalifun'}


In [9]:
target = "add"
agg_result = []
for key in full_org_results.keys():
    a = full_org_results[key][this_year][target]
    b = full_org_results[key][last_year][target]
    delta = 0.00
    if b > 0:
        delta = (-100.0*(b-a)/b)
    temp = {}
    temp["name"] = key
    temp[this_year] = a
    temp[last_year] = b
    temp["change"] = delta
    agg_result.append(temp)
    
newlist = sorted(agg_result, key=lambda d: d[this_year])
newlist.reverse()
print("Results for '{0}' across {1} org".format(target,org))
print("-------------------------------------------------------------------")
for i in newlist:  
    print("{0:24s}| 2023: {1:<8} | 2024: {2:<8} | delta: {3:4.2f}%".format(i["name"][:24],
                                                                           i[last_year],
                                                                           i[this_year],
                                                                           i["change"]))
    
df = pd.DataFrame(data=newlist)
df.to_csv("{0}-{1}-{2}-NewLines.csv".format(org_name,last_year,this_year))

Results for 'add' across Organization(login="open-rmf") org
-------------------------------------------------------------------
rmf_demos               | 2023: 675      | 2024: 2222269  | delta: 329125.04%
rmf_site                | 2023: 14007    | 2024: 519466   | delta: 3608.62%
bevy_impulse            | 2023: 15601    | 2024: 90271    | delta: 478.62%
mapf                    | 2023: 2116     | 2024: 5058     | delta: 139.04%
rmf_traffic             | 2023: 183      | 2024: 3794     | delta: 1973.22%
sdf_rust_experimental   | 2023: 0        | 2024: 3735     | delta: 0.00%
rmf_ros2                | 2023: 3538     | 2024: 3660     | delta: 3.45%
free_fleet              | 2023: 20216    | 2024: 2607     | delta: -87.10%
fleet_adapter_mir       | 2023: 3977     | 2024: 1055     | delta: -73.47%
rmf-web                 | 2023: 7574     | 2024: 694      | delta: -90.84%
rmf_traffic_editor      | 2023: 641      | 2024: 664      | delta: 3.59%
rmf_simulation          | 2023: 1029     | 2024:

In [10]:
target = "prs"

agg_result = []
for key in full_org_results.keys():
    a = len(full_org_results[key][2024][target])
    b = len(full_org_results[key][2023][target])
    delta = 0.00
    if b > 0:
        delta = (-100.0*(b-a)/b)
    temp = {}
    temp["name"] = key
    temp[this_year] = a
    temp[last_year] = b
    temp["change"] = delta
    agg_result.append(temp)
    
newlist = sorted(agg_result, key=lambda d: d[this_year])
newlist.reverse()
print("PR count by year")
print("Results for '{0}' across ROS 2 org".format(target))
print("-------------------------------------------------------------------")
for i in newlist:  
    print("{0:24s}| 2023: {1:<8} | 2024: {2:<8} | delta: {3:4.2f}%".format(i["name"][:24],
                                                                           i[last_year],
                                                                           i[this_year],
                                                                           i["change"]))
df = pd.DataFrame(data=newlist)
df.to_csv("{0}-{1}-{2}-PRS.csv".format(org_name,last_year,this_year))

PR count by year
Results for 'prs' across ROS 2 org
-------------------------------------------------------------------
rmf_site                | 2023: 18       | 2024: 66       | delta: 266.67%
rmf_ros2                | 2023: 18       | 2024: 37       | delta: 105.56%
bevy_impulse            | 2023: 19       | 2024: 35       | delta: 84.21%
rmf_demos               | 2023: 12       | 2024: 26       | delta: 116.67%
free_fleet              | 2023: 18       | 2024: 23       | delta: 27.78%
rmf_traffic_editor      | 2023: 15       | 2024: 13       | delta: -13.33%
rmf_simulation          | 2023: 10       | 2024: 11       | delta: 10.00%
rmf_traffic             | 2023: 4        | 2024: 10       | delta: 150.00%
rmf_internal_msgs       | 2023: 8        | 2024: 8        | delta: -0.00%
rmf-web                 | 2023: 23       | 2024: 8        | delta: -65.22%
rmf                     | 2023: 3        | 2024: 6        | delta: 100.00%
mapf                    | 2023: 2        | 2024: 5        |